# Batched prompt scoring with vLLM

Compute the log-probability of each prompt under vLLM and compare
against the HF Transformers companion notebook
(`test_transformers_batched_prompt_scoring_v1.ipynb`).

vLLM exposes prompt log-probs via
`SamplingParams(prompt_logprobs=N)`. There is no padding-side
concept — the engine handles batching internally.

Note: last-token *embedding* extraction needs a different engine
task (`LLM(..., task="embed")` + `PoolingParams`) and cannot share
this generation engine; covered in a separate notebook if needed.

## Setup

In [ ]:
import torch
from vllm import LLM, SamplingParams

In [ ]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
# llm_dir = base_dir + "Llama-3.2-1B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"

In [ ]:
# Load Qwen2.5-3B-Instruct under vLLM.
# - dtype="auto" picks up the released bf16 weights.
# - gpu_memory_utilization is conservative; raise if you have headroom.
#   vLLM preallocates a KV-cache pool sized from this fraction.
llm_vllm = LLM(
    model=llm_dir,
    dtype="auto",
    gpu_memory_utilization=0.5,
)

# vLLM allocates outside PyTorch's pool, so torch.cuda.memory_allocated
# shows ~0 here. Use mem_get_info() to read driver-level usage.
free, total = torch.cuda.mem_get_info(0)
print(f'#--- GPU memory used: {(total - free) / (1024**3):.2f} GB')

In [ ]:
def sum_token_logprobs(token_ids, logprobs):
    """Sum logprob[tok_id] for matching (token_id, dict) pairs.

    `logprobs` is aligned with `token_ids`. Entries may be None
    (e.g. prompt position 0 has no prior context) and are skipped.
    """
    total = 0.0
    for tok_id, lp_dict in zip(token_ids, logprobs):
        if lp_dict is None:
            continue
        total += lp_dict[tok_id].logprob
    return total

## Test prompts

In [ ]:
# Same prompts as the HF Transformers notebook for cross-reference
texts = [
    "Hello, how are you?",
    "What is your name?",
    "Tell me a joke.",
    "Explain quantum computing in simple terms."
]

## Prompt log-probabilities

Set `prompt_logprobs=N` to have vLLM return per-token log-probs for
the prompt itself. `max_tokens=1` is the minimum allowed; we ignore
the generated token.

Each entry in `RequestOutput.prompt_logprobs` is either `None`
(position 0 — no prior context to condition on) or a dict
`{token_id: Logprob(logprob, rank, decoded_token)}` containing the
top-N tokens *and* the actual prompt token (added if it falls
outside the top-N). Total log-prob = sum over real prompt tokens.

Per-prompt totals should track the HF notebook's batched/single
values modulo backend numerical drift.

In [ ]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1,         # vLLM requires >=1; the generated token is unused
    prompt_logprobs=1,    # enable per-token prompt log-probs
)

outputs = llm_vllm.generate(texts, sampling_params)

for i, output in enumerate(outputs):
    prompt_ids = output.prompt_token_ids
    # Position 0 has no logprob (no prior context to score it against)
    assert output.prompt_logprobs[0] is None

    prompt_lp = sum_token_logprobs(prompt_ids, output.prompt_logprobs)

    print(f"=== Input {i+1}: {output.prompt}")
    print(f"  Scored tokens   : {len(prompt_ids) - 1}")
    print(f"  Prompt log-prob : {prompt_lp:.4f}")
    print()

## Prompt + generation log-probabilities

To score the full sequence (prompt + N generated tokens) in one call,
set **both** `prompt_logprobs` (covers the prompt) and `logprobs`
(covers the continuation). vLLM returns:

- `output.prompt_logprobs` — list aligned with `prompt_token_ids`;
  position 0 is `None` (nothing to condition on).
- `output.outputs[0].logprobs` — list aligned with the generated
  `token_ids`; one entry per generated token.

For greedy decoding (`temperature=0.0`) the sampled token is the
top-1, and `logprobs=1` is the minimum that always reports its
log-prob. Total log-prob = sum of prompt log-probs (positions 1..)
plus sum of generation log-probs.

In [ ]:
N = 20

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=N,
    prompt_logprobs=1,    # per-token log-probs for the prompt
    logprobs=1,           # per-token log-probs for the generated tokens
)

outputs = llm_vllm.generate(texts, sampling_params)

for i, output in enumerate(outputs):
    prompt_ids = output.prompt_token_ids
    gen = output.outputs[0]

    prompt_lp = sum_token_logprobs(prompt_ids, output.prompt_logprobs)
    gen_lp = sum_token_logprobs(gen.token_ids, gen.logprobs)
    total_lp = prompt_lp + gen_lp

    print(f"=== Input {i+1}: {output.prompt}")
    print(f"  Completion          : {gen.text!r}")
    print(f"  Prompt log-prob     : {prompt_lp:.4f} "
          f"({len(prompt_ids) - 1} tokens)")
    print(f"  Generation log-prob : {gen_lp:.4f} "
          f"({len(gen.token_ids)} tokens)")
    print(f"  Combined log-prob   : {total_lp:.4f}")
    print()

## Sampled generation with controlled seed

For diverse continuations (e.g. best-of-N candidate generation),
flip from greedy to sampling: set `temperature > 0`, optionally
`top_p` / `top_k`, and pin `seed` for reproducibility.

`seed` lives on `SamplingParams` — it seeds the per-request RNG.
Re-running the same call with the same seed produces the same
completions; changing the seed re-rolls. With `temperature=0.0`
the seed is ignored (no randomness to control).

`n=K` draws K samples per prompt in a single call; combined with
a fixed seed, this is the reproducible best-of-N pattern.

In [ ]:
SEED = 42
N_SAMPLES = 4

sampling_params = SamplingParams(
    n=N_SAMPLES,
    temperature=0.7,
    top_p=0.9,
    max_tokens=20,
    seed=SEED,
    logprobs=1,           # per-token log-probs for each sampled completion
)

outputs = llm_vllm.generate(texts, sampling_params)

for i, output in enumerate(outputs):
    print(f"=== Input {i+1}: {output.prompt}")
    for k, completion in enumerate(output.outputs):
        gen_lp = sum_token_logprobs(completion.token_ids, completion.logprobs)
        print(f"  Sample {k+1} (log-prob {gen_lp:.4f}): {completion.text!r}")
    print()